In [8]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import polars as pl

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, OrdinalEncoder
from sklearn.metrics import classification_report, f1_score


# Importación de los modelos destacados
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import GradientBoostingClassifier

warnings.filterwarnings('ignore')
print("✅ Librerías cargadas correctamente.")

✅ Librerías cargadas correctamente.


In [ ]:
# 🎯 1. Ruta al dataset VAE seleccionado (Sintético, Latente o Reconstruido)
dataset_path = r"C:\Users\Usuario\Documents\Workspace\Mirage\TFG\dataset\dataset_finales\Undersampling\diagnosticos_combinados.csv"

df = pl.read_csv(dataset_path, separator="|")

seed = int(time.time_ns() % (2**32))
np.random.seed(seed)

# 2. Filtrar por códigos objetivo (Multiclase: F20, F21, etc.)
codes = sorted(["F20", "F21", "F22", "F23", "F25", "F29", "F60.1"])
df = df.filter(pl.col("DIAG PSQ").is_in(codes))

# 3. Separar Características (X) y Target (y)
diag_colms = [col for col in df.columns if col.startswith("Diag")]
df_diag = df.select(diag_colms).fill_null("")

# Convertir X usando OrdinalEncoder (más limpio y estándar para features que LabelEncoder)
X_pandas = df_diag.to_pandas()
encoder_X = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X = encoder_X.fit_transform(X_pandas)

# Convertir y a enteros (0 a 6)
le_y = LabelEncoder()
y = le_y.fit_transform(df.select("DIAG PSQ").to_series().to_numpy())

# 4. División Train / Test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=seed, stratify=y
)

In [10]:
print("🚀 Iniciando Grid Search para LIGHTGBM...")

# Malla de parámetros optimizada para LightGBM
param_grid_lgbm = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 6, 9, -1],
    'num_leaves': [15, 31, 63],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

grid_lgbm = GridSearchCV(
    estimator=LGBMClassifier(random_state=seed, verbose=-1, n_jobs=-1),
    param_grid=param_grid_lgbm,
    scoring='f1_weighted',
    cv=5,
    n_jobs=-1,
    verbose=1
)

t0 = time.time()
grid_lgbm.fit(X_train, y_train)
t_total = time.time() - t0

print(f"\n⏱️ Búsqueda completada en {t_total:.2f} segundos.")
print("🏆 MEJORES HIPERPARÁMETROS (LightGBM):")
print(grid_lgbm.best_params_)

# Evaluación sobre Test
best_lgbm = grid_lgbm.best_estimator_
y_pred_lgbm = best_lgbm.predict(X_test)

print("\n📊 REPORTE DE CLASIFICACIÓN EN TEST (LightGBM):")
print(classification_report(y_test, y_pred_lgbm, target_names=le_y.classes_))

🚀 Iniciando Grid Search para LIGHTGBM...
Fitting 5 folds for each of 432 candidates, totalling 2160 fits

⏱️ Búsqueda completada en 1055.47 segundos.
🏆 MEJORES HIPERPARÁMETROS (LightGBM):
{'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 100, 'num_leaves': 31, 'subsample': 0.8}

📊 REPORTE DE CLASIFICACIÓN EN TEST (LightGBM):
              precision    recall  f1-score   support

         F20       0.59      0.81      0.68       150
         F21       0.00      0.00      0.00         2
         F22       0.42      0.35      0.38        81
         F23       0.17      0.14      0.15        21
         F25       0.52      0.42      0.47        33
         F29       0.31      0.11      0.16        45
       F60.1       0.00      0.00      0.00         4

    accuracy                           0.51       336
   macro avg       0.29      0.26      0.26       336
weighted avg       0.47      0.51      0.47       336



In [11]:
print("🚀 Iniciando Grid Search para XGBOOST...")

# Malla de parámetros optimizada para XGBoost
param_grid_xgb = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.9, 1.0],
    'gamma': [0, 0.1, 0.2]
}

grid_xgb = GridSearchCV(
    estimator=XGBClassifier(random_state=seed, eval_metric='mlogloss', n_jobs=-1),
    param_grid=param_grid_xgb,
    scoring='f1_weighted',
    cv=5,
    n_jobs=-1,
    verbose=1
)

t0 = time.time()
grid_xgb.fit(X_train, y_train)
t_total = time.time() - t0

print(f"\n⏱️ Búsqueda completada en {t_total:.2f} segundos.")
print("🏆 MEJORES HIPERPARÁMETROS (XGBoost):")
print(grid_xgb.best_params_)

# Evaluación sobre Test
best_xgb = grid_xgb.best_estimator_
y_pred_xgb = best_xgb.predict(X_test)

print("\n📊 REPORTE DE CLASIFICACIÓN EN TEST (XGBoost):")
print(classification_report(y_test, y_pred_xgb, target_names=le_y.classes_))

🚀 Iniciando Grid Search para XGBOOST...
Fitting 5 folds for each of 729 candidates, totalling 3645 fits

⏱️ Búsqueda completada en 211.93 segundos.
🏆 MEJORES HIPERPARÁMETROS (XGBoost):
{'colsample_bytree': 0.7, 'gamma': 0, 'learning_rate': 0.05, 'max_depth': 5, 'n_estimators': 200, 'subsample': 0.9}

📊 REPORTE DE CLASIFICACIÓN EN TEST (XGBoost):
              precision    recall  f1-score   support

         F20       0.57      0.79      0.66       150
         F21       0.00      0.00      0.00         2
         F22       0.45      0.41      0.43        81
         F23       0.10      0.05      0.06        21
         F25       0.43      0.36      0.39        33
         F29       0.35      0.13      0.19        45
       F60.1       0.00      0.00      0.00         4

    accuracy                           0.51       336
   macro avg       0.27      0.25      0.25       336
weighted avg       0.46      0.51      0.47       336



In [12]:
print("🚀 Iniciando Grid Search para GRADIENT BOOSTING (Scikit-Learn)...")

# Malla de parámetros optimizada para GradientBoostingClassifier
param_grid_gb = {
    'n_estimators': [100, 200],
    'learning_rate': [0.03, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'subsample': [0.8, 1.0]
}

grid_gb = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=seed),
    param_grid=param_grid_gb,
    scoring='f1_weighted',
    cv=5,
    n_jobs=-1,
    verbose=1
)

t0 = time.time()
grid_gb.fit(X_train, y_train)
t_total = time.time() - t0

print(f"\n⏱️ Búsqueda completada en {t_total:.2f} segundos.")
print("🏆 MEJORES HIPERPARÁMETROS (GradientBoosting):")
print(grid_gb.best_params_)

# Evaluación sobre Test
best_gb = grid_gb.best_estimator_
y_pred_gb = best_gb.predict(X_test)

print("\n📊 REPORTE DE CLASIFICACIÓN EN TEST (GradientBoosting):")
print(classification_report(y_test, y_pred_gb, target_names=le_y.classes_))

🚀 Iniciando Grid Search para GRADIENT BOOSTING (Scikit-Learn)...
Fitting 5 folds for each of 144 candidates, totalling 720 fits

⏱️ Búsqueda completada en 224.27 segundos.
🏆 MEJORES HIPERPARÁMETROS (GradientBoosting):
{'learning_rate': 0.05, 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200, 'subsample': 0.8}

📊 REPORTE DE CLASIFICACIÓN EN TEST (GradientBoosting):
              precision    recall  f1-score   support

         F20       0.56      0.77      0.65       150
         F21       0.00      0.00      0.00         2
         F22       0.39      0.32      0.35        81
         F23       0.12      0.10      0.11        21
         F25       0.45      0.27      0.34        33
         F29       0.33      0.16      0.21        45
       F60.1       0.00      0.00      0.00         4

    accuracy                           0.48       336
   macro avg       0.26      0.23      0.24       336
weighted avg       0.44      0.48      0.44       336

